# webmeta: language, country and register metadata for web data

This notebook turns web data into a research dataset. It finds documents by **language**, **country** and **register** (text variety, such as *news report*, *recipe* or *opinion blog*), then draws a reproducible sample: so many documents from each group you choose, e.g. 50 per language and register.

- **Try it on the demo sample:** a small multilingual sample of [HPLT 4](https://hplt-project.org/), where every document carries language identification, its URL and register probabilities.
- **Or bring your own (small) corpus:** upload a file and tell the notebook which column holds what (step 2).

Run the cells from top to bottom (**Runtime → Run all** works). Every cell is a form: change the values on the right, no code needed. If Colab warns that the notebook was not authored by Google, click **Run anyway**.

*FIN-CLARIAH deliverable D3.3.3 (a), TurkuNLP, University of Turku.*

In [ ]:
#@title 0. Install webmeta
import importlib.util, subprocess, sys
REPO = "https://github.com/TurkuNLP/web-metadata-fin-clariah"
if importlib.util.find_spec("webmeta") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"webmeta[zst] @ git+{REPO}"], check=True)
import webmeta
print("webmeta", webmeta.__version__)

## 1. Choose the data

In [ ]:
#@title 1. Data source
SOURCE = "Demo sample (HPLT 4, multilingual)"  #@param ["Demo sample (HPLT 4, multilingual)", "Upload my own file"]
DEMO_DATASET = "TurkuNLP/TODO-webmeta-demo"  # TODO: the published demo sample

from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if SOURCE.startswith("Demo"):
    ADAPTER = "hplt4"
    if IN_COLAB:
        DATA = DEMO_DATASET
    else:  # running locally from a clone: use the test fixture
        DATA = str(Path("..") / "tests" / "fixtures" / "hplt4_like.jsonl")
else:
    ADAPTER = "mapping"
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        DATA = next(iter(uploaded))
    else:
        DATA = str(Path("..") / "tests" / "fixtures" / "own_corpus.csv")
print("Reading:", DATA)

## 2. Your own corpus: which column holds what

**Skip this step for the demo sample.**

Your file can be JSONL (one JSON object per line), CSV, TSV or Parquet. Keep it small: a few thousand documents is plenty in Colab. The notebook needs these fields; type the name of *your* column next to each. Leave a box empty if your data has no such field: the step that needs it is then skipped, and the rest still work.

| Field | What it is | Needed for |
|---|---|---|
| `text` | the document text | always |
| `url` | the document's web address | country |
| `lang` | language code, e.g. `fi`, `fin` or `fin_Latn` | language |
| `seg_langs` | languages of the document's parts, separated by spaces or commas | finding multilingual documents |
| `register` | register probabilities: either one column per label (`NA`, `nb`, `OP`, …, found automatically) or one column holding a JSON object such as `{"NA": 0.9, "nb": 0.7}` | register |

Register labels follow the [TurkuNLP web register taxonomy](https://turkunlp.org/register-annotation-docs/). Your data needs no register probabilities to use the language and country steps.

In [ ]:
#@title 2. Field mapping (own corpus only)
TEXT_COLUMN = "body"  #@param {type:"string"}
URL_COLUMN = "link"  #@param {type:"string"}
LANG_COLUMN = "language"  #@param {type:"string"}
SEG_LANGS_COLUMN = ""  #@param {type:"string"}
REGISTER_COLUMN = ""  #@param {type:"string"}
ID_COLUMN = "doc_id"  #@param {type:"string"}

MAPPING = {"text": TEXT_COLUMN, "url": URL_COLUMN, "lang": LANG_COLUMN,
           "seg_langs": SEG_LANGS_COLUMN, "register": REGISTER_COLUMN, "id": ID_COLUMN}
MAPPING = {k: v for k, v in MAPPING.items() if v}

import itertools
def load(n=None):
    raw = webmeta.read(DATA)
    if n is not None:
        raw = itertools.islice(raw, n)
    if ADAPTER == "hplt4":
        return [webmeta.from_hplt4(r) for r in raw]
    return [webmeta.from_mapping(r, MAPPING) for r in raw]

records = load()
cov = webmeta.coverage(records)
print(f"{cov['records']} documents loaded.")
for step in ("language", "country", "register"):
    share = cov[step] / max(cov["records"], 1)
    print(f"  {step:9s} step possible for {cov[step]:>6} ({share:.0%})")

## 3. Choose what to keep

Leave a box empty to keep everything on that dimension.

- **Languages:** codes separated by commas, e.g. `fi,sv`. Tick *any segment* to keep documents that *contain* the language anywhere, not only those written mainly in it.
- **Countries:** two-letter codes, e.g. `FI,SE`. The country comes from the web address (`.fi` → Finland). Addresses like `.com` give no country.
- **Registers:** labels, e.g. `NA` (narrative) or `nb` (narrative blog). A document can have several; a *hybrid* has two or more main registers.
- **Threshold:** how sure the register classifier must be before a label counts. By default each language uses the value tuned for the classifier (English 0.40, Finnish and Swedish 0.45, …); untick *model thresholds* to set one value for all.
- **Exclude registers:** labels that drop a document, e.g. `MT` for machine-translated text.

In [ ]:
#@title 3. Filters
LANGUAGES = "fi,sv"  #@param {type:"string"}
ANY_SEGMENT = False  #@param {type:"boolean"}
MULTILINGUAL = "either"  #@param ["either", "only multilingual", "only monolingual"]
COUNTRIES = ""  #@param {type:"string"}
REGISTERS = ""  #@param {type:"string"}
HYBRIDS = "either"  #@param ["either", "only hybrids", "only single-register"]
EXCLUDE_REGISTERS = ""  #@param {type:"string"}
MODEL_THRESHOLDS = True  #@param {type:"boolean"}
THRESHOLD = 0.4  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

def _codes(s):
    return [c.strip() for c in s.split(",") if c.strip()] or None

FILTERS = dict(
    lang=_codes(LANGUAGES), any_segment=ANY_SEGMENT,
    multilingual={"either": None, "only multilingual": True, "only monolingual": False}[MULTILINGUAL],
    country=_codes(COUNTRIES), register=_codes(REGISTERS),
    hybrid={"either": None, "only hybrids": True, "only single-register": False}[HYBRIDS],
    exclude=_codes(EXCLUDE_REGISTERS) or (),
)
selected = list(webmeta.run(records, threshold=None if MODEL_THRESHOLDS else THRESHOLD, **FILTERS))
print(f"{len(selected)} of {len(records)} documents kept.")

## 4. Look at the result

In [ ]:
#@title 4. Results table and counts
import pandas as pd
from webmeta.register import NAMES

rows = [{"id": r.get("id"), "language": r["meta"]["lang"], "country": r["meta"]["country"],
         "registers": " + ".join(r["meta"]["registers"]), "multilingual": r["meta"]["multilingual"],
         "url": r.get("url"), "text": (r.get("text") or "")[:200]} for r in selected]
table = pd.DataFrame(rows)
if table.empty:
    print("Nothing matched. Loosen the filters in step 3.")
else:
    print("Documents by language and country:")
    print(table.groupby(["language", "country"], dropna=False).size().to_string())
    counts = webmeta.summary(selected, by=("registers",))
    print("\nDocuments by register label:")
    for (label,), n in counts.most_common():
        print(f"  {label or '(none)':4s} {NAMES.get(label, ''):40s} {n}")
table.head(20)

## 5. Build a dataset

Draw a sample from the documents kept in step 3.

- **Cells:** the groups to sample from, e.g. *language* and *main register*: every combination is a cell.
- **Per cell:** how many documents to draw from each cell. A cell with fewer keeps all it has. Leave 0 to take everything.
- **Minimum words:** drop shorter documents first (0 = keep all).
- **Seed:** the same seed gives the same sample; change it to draw a different one.

The table shows, for each cell, how many documents were available and how many were taken.

In [ ]:
#@title 5. Sample
CELL_1 = "language"  #@param ["language", "country", "main register", "hybrid", "multilingual", "(none)"]
CELL_2 = "main register"  #@param ["language", "country", "main register", "hybrid", "multilingual", "(none)"]
PER_CELL = 50  #@param {type:"integer"}
MIN_WORDS = 0  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}

FIELD = {"language": "lang", "country": "country", "main register": "main_register",
         "hybrid": "hybrid", "multilingual": "multilingual"}
STRATA = tuple(dict.fromkeys(FIELD[c] for c in (CELL_1, CELL_2) if c in FIELD))

fields = {
    "words": lambda r: len((r.get("text") or "").split()),
    "main_register": lambda r: next((l for l in r["meta"]["registers"] if l.isupper()), "none"),
}
docs = webmeta.derive(selected, fields)
docs = webmeta.keep(docs, [lambda r: r["meta"]["words"] >= MIN_WORDS])
sample = webmeta.stratify(docs, strata=STRATA, n=PER_CELL or None, seed=SEED)

cells = pd.DataFrame(list(sample.table()))
print(f"{len(sample)} documents in {len(sample.population)} cells.")
cells

## 6. Download

Choose the format: **jsonl** keeps every field with the metadata webmeta added; **csv** and **parquet** are tables; **txt** is one text file per document. The files are grouped into folders by the cells, zipped, and downloaded.

In [ ]:
#@title 6. Download the dataset
FORMAT = "jsonl"  #@param ["jsonl", "csv", "txt", "parquet"]
OUT = "webmeta_dataset"

import shutil
shutil.rmtree(OUT, ignore_errors=True)
dirs = "/".join("{" + f + "}" for f in STRATA) or None
n = webmeta.export(sample, OUT, format=FORMAT, dirs=dirs,
                   name="{id}_{main_register}", columns=("id", "url", *STRATA, "words", "register"))
archive = shutil.make_archive(OUT, "zip", OUT)
print(f"Wrote {n} documents ({FORMAT}) to {archive}, with cells.tsv: per cell, available and taken.")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

---
FIN-CLARIAH project has received funding from the European Union – NextGenerationEU instrument and is funded by the Research Council of Finland under grant number 358720.